# Experimento 1

In [1]:
pip install open_clip_torch torch torchvision

In [2]:
!git clone https://github.com/StevenFAP1815/Proyecto1_VC_BioCLIP.git

# Cambiar el directorio de trabajo a la raíz de tu proyecto
import os
os.chdir("Proyecto1_VC_BioCLIP")

fatal: destination path 'Proyecto1_VC_BioCLIP' already exists and is not an empty directory.


# 1: Cargar librerías y configurar el dispositivo (CPU/GPU)

In [3]:
import json
import torch
import open_clip
from PIL import Image

# Configurar el dispositivo (usará GPU si está disponible, si no CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


# 2: Cargar el modelo BioCLIP y sus transformaciones

In [4]:
# Cargar el modelo BioCLIP preentrenado desde HuggingFace/OpenCLIP
model, _, preprocess = open_clip.create_model_and_transforms(
    'hf-hub:imageomics/bioclip',
    device=device
)
tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip')

model.eval()
print("Modelo BioCLIP cargado exitosamente.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


Modelo BioCLIP cargado exitosamente.


# 3: Cargar las especies y plantillas desde species_config.json

In [5]:
!git pull origin main

From https://github.com/StevenFAP1815/Proyecto1_VC_BioCLIP
 * branch            main       -> FETCH_HEAD
Already up to date.


In [6]:
# Cargar el archivo de configuración
with open('config/species_config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

species_list = config['species']
templates = config['templates']

# Generar los prompts para cada especie
prompts = []
labels = []

for sp in species_list:
    labels.append(sp['common_name'])
    # Creamos un prompt combinando la plantilla con el nombre científico
    prompt_text = templates[0].format(scientific_name=sp['scientific_name'])
    prompts.append(prompt_text)

print("Prompts generados para el modelo:")
for l, p in zip(labels, prompts):
    print(f"- {l}: '{p}'")

Prompts generados para el modelo:
- Tangerine: 'a photo of a seed of Citrus reticulata'
- Sunflower: 'a photo of a seed of Helianthus annuus'
- Corn: 'a photo of a seed of Zea mays'
- Bean: 'a photo of a seed of Phaseolus vulgaris'
- Tomato: 'a photo of a seed of Solanum lycopersicum'
- Pea: 'a photo of a seed of Pisum sativum'
- Soy: 'a photo of a seed of Glycine max'
- Wheat: 'a photo of a seed of Triticum aestivum'
- Coffe bean: 'a photo of a seed of Coffea arabica'
- Apple: 'a photo of a seed of Malus domestica'


# 4: Evaluar la imagen de la mandarina y calcular probabilidades

In [7]:
import os
print(os.listdir("data/processed_examples"))

['mandarina_01.jpg', 'mandarina_04.jpg']


In [9]:
# 1. Cargar la imagen procesada
image_path = "data/processed_examples/mandarina_04.jpg"
image = Image.open(image_path).convert("RGB")

# 2. Preprocesar la imagen y tokenizar los textos
image_input = preprocess(image).unsqueeze(0).to(device)
text_inputs = tokenizer(prompts).to(device)

# 3. Calcular similitud (Inferencia Zero-Shot)
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)

    # Normalizar vectores
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Calcular similitud coseno multiplicada por el factor de escala del modelo
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    probs = similarity[0].cpu().numpy()

# 4. Mostrar resultados ordenados por porcentaje de confianza
print("\n--- RESULTADOS DE CLASIFICACIÓN ---")
results = sorted(zip(labels, probs), key=lambda x: x[1], reverse=True)
for label, prob in results:
    print(f"{label}: {prob * 100:.2f}%")


--- RESULTADOS DE CLASIFICACIÓN ---
Corn: 65.90%
Coffe bean: 26.61%
Pea: 4.66%
Sunflower: 1.86%
Wheat: 0.68%
Apple: 0.14%
Soy: 0.11%
Bean: 0.01%
Tomato: 0.01%
Tangerine: 0.01%
